In [1]:
import numpy as np
import torch
from torch.utils.data import DataLoader
# from utils import evaluate, ScenarioWiseSampler
import os
from torch.utils.data import Subset
from model import DayCentModel
import random
import matplotlib.pyplot as plt
import os
from data import DayCentDataset

In [2]:
EXPERIMENT_ID = "experiment11"
PROCESSED_DIR = f"/users/6/mehta423/daycent/data/{EXPERIMENT_ID}"
INPUT_NPY = f"/users/6/mehta423/daycent/data/{EXPERIMENT_ID}/test_X.npy"
OUTPUT_NPY = f"/users/6/mehta423/daycent/data/{EXPERIMENT_ID}/test_Y.npy"
INIT_COND = "/users/6/mehta423/daycent/data/SAS_KGML_090925/InputData/initial_site_conditions.xlsx"
OUTPUT_DIR = f"/users/6/mehta423/daycent/output/{EXPERIMENT_ID}"
PLOTS_DIR = f"/users/6/mehta423/daycent/output/{EXPERIMENT_ID}/plots"
BATCH_SIZE = 2048
EPOCHS = 100
DEVICE = torch.device("cuda:3" if torch.cuda.is_available() else "cpu")

In [3]:
# ----------------------
# Dataloader
# ----------------------
# check dataset
dataset = DayCentDataset(INPUT_NPY, OUTPUT_NPY, INIT_COND, apply_scaling=True)
NUM_SCENARIOS = 100
UNIT_SIZE = int(len(dataset) / NUM_SCENARIOS)
train_size = int(UNIT_SIZE * (NUM_SCENARIOS * 0.7))
val_size = int(UNIT_SIZE * (NUM_SCENARIOS * 0.2))
test_size = int(UNIT_SIZE * (NUM_SCENARIOS * 0.1))

# 2) Create index arrays for each split
train_idx = np.arange(0, train_size)
val_idx = np.arange(train_size, train_size + val_size)
test_idx = np.arange(train_size + val_size, train_size + val_size + test_size)
all_idx = np.arange(0, train_size + val_size + test_size)                     

# 3) Wrap subsets
train_ds = Subset(dataset, train_idx)
val_ds   = Subset(dataset, val_idx)
test_ds  = Subset(dataset, test_idx)

# 4) Create loaders
all_loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=4)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=4)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

print(f"Dataset sizes — total: {len(dataset)}, train: {len(train_ds)}, val: {len(val_ds)}, test: {len(test_ds)}")

0
Dataset sizes — total: 367500, train: 257250, val: 73500, test: 36750


In [4]:
# infer input dim
sample = dataset[0]
seq_feat_dim = sample["sequence"].shape[1]  # #features
init_dim = sample["init_cond"].shape[0]
year_dim = sample["year_enc"].shape[0]

print(f"Input feature dim: {seq_feat_dim}, init cond dim: {init_dim}, year enc dim: {year_dim}")


model = DayCentModel(input_dim=seq_feat_dim, init_dim=init_dim, year_dim=year_dim)
model.load_state_dict(torch.load(os.path.join(OUTPUT_DIR, "best_model.pth")))
model.to(DEVICE)

Input feature dim: 20, init cond dim: 245, year enc dim: 16


DayCentModel(
  (init_proj): Linear(in_features=261, out_features=32, bias=True)
  (daily_proj): Linear(in_features=20, out_features=32, bias=True)
  (lstm): LSTM(64, 128, num_layers=2, batch_first=True)
  (somsc_attn): AttentionPooling(
    (attn): Linear(in_features=128, out_features=1, bias=True)
    (proj): Linear(in_features=128, out_features=128, bias=True)
  )
  (yield_attn): AttentionPooling(
    (attn): Linear(in_features=128, out_features=1, bias=True)
    (proj): Linear(in_features=128, out_features=128, bias=True)
  )
  (somsc_head): Linear(in_features=128, out_features=1, bias=True)
  (yield_head): Linear(in_features=128, out_features=1, bias=True)
)

In [5]:
all_preds = []
all_trues = []
all_masks = []
for batch in all_loader:
    batch = {k: v.to(DEVICE) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}
    with torch.no_grad():
        outputs = model(batch)

    all_preds.append(outputs['yield_pred'].cpu().numpy())
    all_trues.append(batch['yield'].cpu().numpy())
    all_masks.append(batch['yield_mask'].cpu().numpy())

# Concatenate all batches
pred_flat = np.concatenate(all_preds, axis=0)
true_flat = np.concatenate(all_trues, axis=0)
mask_flat = np.concatenate(all_masks, axis=0)

In [6]:
import joblib
scaler_Y = joblib.load(os.path.join(PROCESSED_DIR, "scaler_Y.pkl"))
cgrain_mean = scaler_Y.mean_[1]
cgrain_scale = scaler_Y.scale_[1]

# Inverse transform manually
pred_flat = pred_flat * cgrain_scale + cgrain_mean
pred_flat = pred_flat * mask_flat
true_flat = true_flat * cgrain_scale + cgrain_mean
true_flat = true_flat * mask_flat

In [14]:
class ScenarioWiseSampler:
    def __init__(self, input_path: str, indices):
        data_dict = np.load(input_path, allow_pickle=True).item()
        self.data = data_dict["data"]      # (N, 365, #features)
        self.mapping = data_dict["mapping"][indices]  # (N, 3) => (scenario, point_id, year)
        self.columns = list(data_dict["columns"])

    def get_year_indices(self, sid, pid):
        # get indices for all years for a given scenario id and plot id
        mask = (self.mapping[:, 0] == sid) & (self.mapping[:, 2] == pid)
        return np.where(mask)[0]
    
    def __iter__(self):
        for sid, year, pid in self.mapping:
            yield sid, year, pid

    def get_all_unique_pids(self):
        return np.unique(self.mapping[:, 2])
    
    def get_all_unique_scenarios(self):
        unique_scenarios = np.unique(self.mapping[:, 0])
        return unique_scenarios

    def __len__(self) -> int:
        return len(self.mapping)

all_sampler = ScenarioWiseSampler(INPUT_NPY, all_idx)
# train_sampler = ScenarioWiseSampler(INPUT_NPY, train_idx)
# test_sampler = ScenarioWiseSampler(INPUT_NPY, test_idx)
# val_sampler = ScenarioWiseSampler(INPUT_NPY, val_idx)

In [8]:
sids = all_sampler.get_all_unique_scenarios()
pids = all_sampler.get_all_unique_pids()

len(sids), len(pids)

Unique Scenarios in the sampler:
scenario_10
scenario_1041
scenario_1085
scenario_1134
scenario_1170
scenario_1291
scenario_1404
scenario_1490
scenario_1744
scenario_1797
scenario_1828
scenario_1833
scenario_1877
scenario_189
scenario_2267
scenario_2288
scenario_2341
scenario_2505
scenario_2592
scenario_2622
scenario_2647
scenario_2678
scenario_2804
scenario_2928
scenario_320
scenario_3259
scenario_3433
scenario_3457
scenario_3484
scenario_3594
scenario_3599
scenario_3753
scenario_3924
scenario_3947
scenario_4011
scenario_4041
scenario_4120
scenario_4305
scenario_4316
scenario_4340
scenario_4372
scenario_4375
scenario_4387
scenario_4423
scenario_4809
scenario_4890
scenario_5039
scenario_5156
scenario_5169
scenario_526
scenario_5311
scenario_5314
scenario_54
scenario_5574
scenario_5821
scenario_5931
scenario_5948
scenario_6066
scenario_6127
scenario_6217
scenario_6253
scenario_6305
scenario_6483
scenario_6544
scenario_6573
scenario_6917
scenario_7020
scenario_7124
scenario_7434
scenario

(100, 147)

In [9]:
for sid in sids[-5:]:
    if not os.path.exists(os.path.join(PLOTS_DIR, sid)):
        os.makedirs(os.path.join(PLOTS_DIR, sid))
    random_pids = random.sample(list(pids), 5)
    for i, pid in enumerate(random_pids):
        year_indices = all_sampler.get_year_indices(sid,pid)
        pred = (pred_flat * mask_flat)[year_indices]
        true = true_flat[year_indices]

        # Assuming pred and true are numpy arrays (or lists)
        years = np.arange(2000, 2025)

        plt.figure(figsize=(10, 5))
        plt.plot(years, true, marker='o', label='True', linewidth=2)
        plt.plot(years, pred, marker='s', label='Predicted', linewidth=2, linestyle='--')

        plt.title('Predicted vs True Values (2000–2024)')
        plt.xlabel('Year')
        plt.ylabel('Value')
        plt.legend()
        plt.grid(True, linestyle='--', alpha=0.6)
        plt.tight_layout()
        plt.savefig(os.path.join(PLOTS_DIR, sid, f'{pid}.png'))
        # plt.show()
        plt.close()


In [17]:
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from scipy.stats import pearsonr
import pandas as pd
from collections import defaultdict

def compute_metrics_vectorized(true_flat, pred_flat, mask_flat, all_sampler):
    """
    Vectorized computation of metrics for all (scenario, point) combinations
    """
    sids = all_sampler.get_all_unique_scenarios()
    pids = all_sampler.get_all_unique_pids()
    
    # Pre-compute all masks for each (sid, pid) combination
    mapping = all_sampler.mapping
    
    # Create a dictionary to store indices for each (sid, pid)
    # This is much faster than calling get_year_indices repeatedly
    indices_dict = defaultdict(list)
    for idx, (sid, year, pid) in enumerate(mapping):
        indices_dict[(sid, pid)].append(idx)
    
    # Convert to numpy arrays for vectorized operations
    results = {
        'scenario': [],
        'point_id': [],
        'r2': [],
        'mae': [],
        'rmse': [],
        'pearson_r': [],
        'nse': [],
        'nrmse': [],
        'n_years': []
    }
    
    # Process all combinations
    for (sid, pid), year_indices in indices_dict.items():
        year_indices = np.array(year_indices)
        
        if len(year_indices) == 0:
            continue
        
        pred = (pred_flat * mask_flat)[year_indices]
        true = (true_flat * mask_flat)[year_indices]
        
        # Skip if all values are the same (causes issues with correlation)
        if np.std(true) == 0 or np.std(pred) == 0:
            continue
        
        # Vectorized computations
        diff = true - pred
        
        # MAE
        mae = np.mean(np.abs(diff))
        
        # RMSE
        rmse = np.sqrt(np.mean(diff**2))
        
        # R²
        ss_res = np.sum(diff**2)
        ss_tot = np.sum((true - np.mean(true))**2)
        r2 = 1 - (ss_res / ss_tot) if ss_tot != 0 else np.nan
        
        # Pearson correlation (vectorized)
        true_centered = true - np.mean(true)
        pred_centered = pred - np.mean(pred)
        pearson_r = np.sum(true_centered * pred_centered) / (
            np.sqrt(np.sum(true_centered**2)) * np.sqrt(np.sum(pred_centered**2))
        )
        
        # NSE
        nse = 1 - (ss_res / ss_tot) if ss_tot != 0 else np.nan
        
        # NRMSE
        nrmse = (rmse / np.mean(true) * 100) if np.mean(true) != 0 else np.nan
        
        # Store results
        results['scenario'].append(sid)
        results['point_id'].append(pid)
        results['r2'].append(r2)
        results['mae'].append(mae)
        results['rmse'].append(rmse)
        results['pearson_r'].append(pearson_r)
        results['nse'].append(nse)
        results['nrmse'].append(nrmse)
        results['n_years'].append(len(year_indices))
    
    return pd.DataFrame(results)


# Use the vectorized function
print("Computing metrics...")
df_results = compute_metrics_vectorized(true_flat, pred_flat, mask_flat, all_sampler)

# ============================================
# AGGREGATE BY SCENARIO
# ============================================
scenario_metrics = df_results.groupby('scenario').agg({
    'r2': ['mean', 'std', 'median', 'min', 'max'],
    'mae': ['mean', 'std', 'median'],
    'rmse': ['mean', 'std', 'median'],
    'pearson_r': ['mean', 'std', 'median'],
    'nse': ['mean', 'std', 'median'],
    'nrmse': ['mean', 'std', 'median'],
    'n_years': 'first'  # Should be same for all in a scenario
}).round(4)

print("=" * 80)
print("METRICS AGGREGATED BY SCENARIO")
print("=" * 80)
print(scenario_metrics)

# ============================================
# AGGREGATE BY SPATIAL POINT
# ============================================
point_metrics = df_results.groupby('point_id').agg({
    'r2': ['mean', 'std', 'median', 'min', 'max'],
    'mae': ['mean', 'std', 'median'],
    'rmse': ['mean', 'std', 'median'],
    'pearson_r': ['mean', 'std', 'median'],
    'nse': ['mean', 'std', 'median'],
    'nrmse': ['mean', 'std', 'median']
}).round(4)

print("\n" + "=" * 80)
print("METRICS AGGREGATED BY SPATIAL POINT")
print("=" * 80)
print(point_metrics.head(20))  # Show first 20 points

# ============================================
# OVERALL METRICS
# ============================================
overall_metrics = df_results[['r2', 'mae', 'rmse', 'pearson_r', 'nse', 'nrmse']].agg(
    ['mean', 'std', 'median', 'min', 'max']
).round(4)

print("\n" + "=" * 80)
print("OVERALL METRICS")
print("=" * 80)
print(overall_metrics)

print(f"\nTotal combinations evaluated: {len(df_results)}")

Computing metrics...
METRICS AGGREGATED BY SCENARIO
                   r2                                       mae           \
                 mean     std  median      min     max     mean      std   
scenario                                                                   
scenario_10   -0.7028  2.5347  0.8369  -8.5419  0.9718  73.2122  73.7832   
scenario_1041  0.4453  0.8092  0.8816  -2.6190  0.9815  58.4772  51.9874   
scenario_1085 -2.4668  5.1420  0.5903 -19.9431  0.9559  91.1603  92.3627   
scenario_1134  0.2977  1.0534  0.8858  -3.5147  0.9772  62.1893  55.7094   
scenario_1170 -1.9998  4.5070  0.4940 -20.7942  0.9446  84.2436  79.9203   
...               ...     ...     ...      ...     ...      ...      ...   
scenario_9764 -0.1533  1.6392  0.8122  -5.7181  0.9653  75.3769  68.0939   
scenario_9772 -0.0361  1.5116  0.8480  -5.3113  0.9665  71.2671  68.0412   
scenario_9814  0.2612  1.0167  0.8598  -3.3327  0.9842  64.9517  55.1276   
scenario_9978 -0.0774  1.6479  0.855

In [18]:
scenario_metrics

r2                                       mae           \
                 mean     std  median      min     max     mean      std   
scenario                                                                   
scenario_10   -0.7028  2.5347  0.8369  -8.5419  0.9718  73.2122  73.7832   
scenario_1041  0.4453  0.8092  0.8816  -2.6190  0.9815  58.4772  51.9874   
scenario_1085 -2.4668  5.1420  0.5903 -19.9431  0.9559  91.1603  92.3627   
scenario_1134  0.2977  1.0534  0.8858  -3.5147  0.9772  62.1893  55.7094   
scenario_1170 -1.9998  4.5070  0.4940 -20.7942  0.9446  84.2436  79.9203   
...               ...     ...     ...      ...     ...      ...      ...   
scenario_9764 -0.1533  1.6392  0.8122  -5.7181  0.9653  75.3769  68.0939   
scenario_9772 -0.0361  1.5116  0.8480  -5.3113  0.9665  71.2671  68.0412   
scenario_9814  0.2612  1.0167  0.8598  -3.3327  0.9842  64.9517  55.1276   
scenario_9978 -0.0774  1.6479  0.8559  -6.5264  0.9603  70.9029  68.0399   
scenario_9981 -2.3743  5.0896  0.7557 -17.2790  0.9235  85.8763  87.0715   

                            rmse           ... pearson_r                  \
                median      mean      std  ...      mean     std  median   
scenario                                   ...                             
scenario_10    26.8391   91.1733  83.1515  ...    0.9010  0.0911  0.9483   
scenario_1041  33.0071   74.9977  59.9608  ...    0.9431  0.0573  0.9766   
scenario_1085  35.9897  101.5371  95.6147  ...    0.8029  0.1897  0.9263   
scenario_1134  34.6693   79.1218  65.1879  ...    0.9178  0.0751  0.9627   
scenario_1170  38.3408   98.0482  82.2550  ...    0.8077  0.1675  0.9065   
...                ...       ...      ...  ...       ...     ...     ...   
scenario_9764  38.2124   89.9223  73.2444  ...    0.8408  0.1665  0.9525   
scenario_9772  30.7272   87.6302  73.8537  ...    0.8678  0.1336  0.9568   
scenario_9814  35.4731   84.9735  62.8932  ...    0.9153  0.0680  0.9531   
scenario_9978  33.0616   84.9790  73.8572  ...    0.8786  0.1246  0.9583   
scenario_9981  32.7717   97.7665  90.8005  ...    0.7774  0.1997  0.9144   

                  nse                    nrmse                   n_years  
                 mean     std  median     mean      std   median   first  
scenario                                                                  
scenario_10   -0.7028  2.5347  0.8369  66.5979  62.5233  27.4112      25  
scenario_1041  0.4453  0.8092  0.8816  37.2574  30.7690  22.0731      25  
scenario_1085 -2.4668  5.1420  0.5903  61.0589  58.8309  27.4048      25  
scenario_1134  0.2977  1.0534  0.8858  44.0189  37.6862  23.3951      25  
scenario_1170 -1.9998  4.5070  0.4940  53.0776  45.6801  28.3932      25  
...               ...     ...     ...      ...      ...      ...     ...  
scenario_9764 -0.1533  1.6392  0.8122  38.3645  32.3714  20.3138      25  
scenario_9772 -0.0361  1.5116  0.8480  42.2356  36.4648  20.8373      25  
scenario_9814  0.2612  1.0167  0.8598  43.3795  33.5813  24.3328      25  
scenario_9978 -0.0774  1.6479  0.8559  43.6453  38.8641  21.6688      25  
scenario_9981 -2.3743  5.0896  0.7557  56.4927  53.8781  21.2222      25  

[100 rows x 21 columns]